In [ ]:
# instalando as bibliotecas necessárias
!pip install -q pyspark
!pip install -q findspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Importando as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark import SparkContext, SQLContext
from pyspark.sql.functions import col, split, lit, regexp_replace, rand, when, udf, isnull, size, monotonically_increasing_id, lower, trim
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import random

In [ ]:
# iniciando sessão Spark
spark = SparkSession.builder.appName("mySparkSession").getOrCreate()

In [ ]:
!unzip Filmes+e+Series_backup.zip

Archive:  Filmes+e+Series_backup.zip
  inflating: movies.csv              
  inflating: series.csv              


In [ ]:
# Baixando o arquivo e lendo as primeiras linhas
df_movie_uniao = spark.read.csv("/content/movies.csv", header= True, inferSchema=True, sep="|")
df_serie_uniao = spark.read.csv("/content/series.csv", header= True, inferSchema=True, sep="|")
df_movie_uniao.show(5)
df_serie_uniao.show(5)

+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|tempoMinutos| genero|notaMedia|numeroVotos|generoArtista|          personagem|      nomeArtista|anoNascimento|anoFalecimento|   profissao|titulosMaisConhecidos|
+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        200|      actress|Miss Geraldine Ho...|  Blanche Bayliss|         1878|          1951|     actress|            tt0000009|
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        200|        actor|        Mr. Hamilton|Wil

In [ ]:
df_movie_series = df_movie_uniao.unionByName(df_serie_uniao, allowMissingColumns=True)
df_movie_series.show(5)

+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+----------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|tempoMinutos| genero|notaMedia|numeroVotos|generoArtista|          personagem|      nomeArtista|anoNascimento|anoFalecimento|   profissao|titulosMaisConhecidos|anoTermino|
+---------+--------------+--------------+-------------+------------+-------+---------+-----------+-------------+--------------------+-----------------+-------------+--------------+------------+---------------------+----------+
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|        200|      actress|Miss Geraldine Ho...|  Blanche Bayliss|         1878|          1951|     actress|            tt0000009|      null|
|tt0000009|    Miss Jerry|    Miss Jerry|         1894|          45|Romance|      5.3|      

In [ ]:
# Cria a visualização temporária
df_movie_series.createOrReplaceTempView("limpeza")

# Consulta SQL - Corrigido
consulta_sql = '''
SELECT *
FROM limpeza
WHERE (genero LIKE '%War%' OR genero LIKE '%Crime%')
  AND anoLancamento >= 2000
  AND anoNascimento >= 1944
'''

# Executa a consulta
df_filtered = spark.sql(consulta_sql)
df_filtered.show()

+---------+-------------------+-------------------+-------------+------------+--------------------+---------+-----------+-------------+--------------------+-------------------+-------------+--------------+--------------------+---------------------+----------+
|       id|     tituloPincipal|     tituloOriginal|anoLancamento|tempoMinutos|              genero|notaMedia|numeroVotos|generoArtista|          personagem|        nomeArtista|anoNascimento|anoFalecimento|           profissao|titulosMaisConhecidos|anoTermino|
+---------+-------------------+-------------------+-------------+------------+--------------------+---------+-----------+-------------+--------------------+-------------------+-------------+--------------+--------------------+---------------------+----------+
|tt0079644|      November 1828|      November 1828|         2001|         140|           Drama,War|      6.8|        214|        actor|     Captain DeBorst|    Slamet Rahardjo|         1949|            \N|actor,director,

In [ ]:
quantidade_de_linhas = df_filtered.count()
print(f"O DataFrame tem {quantidade_de_linhas} linhas.")

O DataFrame tem 47506 linhas.


In [ ]:
# Limpando valores nulos
@udf(returnType=StringType())
def replace_nan(value):
    if value == '\\N':
        return 'sem informação'
    else:
        return value

# Aplicando a função às colunas necessárias
df_filtered = df_filtered.withColumn("tituloPincipal", replace_nan(df_movie_series["tituloPincipal"]))
df_filtered = df_filtered.withColumn("anoLancamento", replace_nan(df_movie_series["anoLancamento"]))
df_filtered = df_filtered.withColumn("anoFalecimento", replace_nan(df_movie_series["anoFalecimento"]))
df_filtered = df_filtered.withColumn("personagem", replace_nan(df_movie_series["personagem"]))

In [ ]:
# Tabela temporária
df_filtered.createOrReplaceTempView("artistas_vivos")

# Consulta SQL para contar o número de pessoas em cada geração
consulta_sql = """
SELECT *
FROM artistas_vivos
WHERE anoFalecimento = 'sem informação';
"""

# Executa a consulta
df_filtered = spark.sql(consulta_sql)
df_filtered.show()

+---------+-------------------+-------------------+-------------+------------+--------------------+---------+-----------+-------------+--------------------+-------------------+-------------+--------------+--------------------+---------------------+----------+
|       id|     tituloPincipal|     tituloOriginal|anoLancamento|tempoMinutos|              genero|notaMedia|numeroVotos|generoArtista|          personagem|        nomeArtista|anoNascimento|anoFalecimento|           profissao|titulosMaisConhecidos|anoTermino|
+---------+-------------------+-------------------+-------------+------------+--------------------+---------+-----------+-------------+--------------------+-------------------+-------------+--------------+--------------------+---------------------+----------+
|tt0079644|      November 1828|      November 1828|         2001|         140|           Drama,War|      6.8|        214|        actor|     Captain DeBorst|    Slamet Rahardjo|         1949|sem informação|actor,director,

In [ ]:
df_filtered.show(5)

+---------+--------------+--------------+-------------+------------+------------------+---------+-----------+-------------+-------------------+-------------------+-------------+--------------+--------------------+---------------------+----------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|tempoMinutos|            genero|notaMedia|numeroVotos|generoArtista|         personagem|        nomeArtista|anoNascimento|anoFalecimento|           profissao|titulosMaisConhecidos|anoTermino|
+---------+--------------+--------------+-------------+------------+------------------+---------+-----------+-------------+-------------------+-------------------+-------------+--------------+--------------------+---------------------+----------+
|tt0079644| November 1828| November 1828|         2001|         140|         Drama,War|      6.8|        214|        actor|    Captain DeBorst|    Slamet Rahardjo|         1949|sem informação|actor,director,wr...| tt1136918,tt02990...|      null|
|tt0079644| 

In [ ]:
# Cria a visualização temporária
df_filtered.createOrReplaceTempView("remover_coluna")

# Consulta SQL
consulta_sql = '''
SELECT id, tituloPincipal, tituloOriginal, anoLancamento, genero, numeroVotos,
       generoArtista, personagem, nomeArtista, anoNascimento, profissao
FROM remover_coluna;
'''

# Executa a consulta
df_filtered = spark.sql(consulta_sql)
df_filtered.show()

+---------+-------------------+-------------------+-------------+--------------------+-----------+-------------+--------------------+-------------------+-------------+--------------------+
|       id|     tituloPincipal|     tituloOriginal|anoLancamento|              genero|numeroVotos|generoArtista|          personagem|        nomeArtista|anoNascimento|           profissao|
+---------+-------------------+-------------------+-------------+--------------------+-----------+-------------+--------------------+-------------------+-------------+--------------------+
|tt0079644|      November 1828|      November 1828|         2001|           Drama,War|        214|        actor|     Captain DeBorst|    Slamet Rahardjo|         1949|actor,director,wr...|
|tt0079644|      November 1828|      November 1828|         2001|           Drama,War|        214|      actress|               Laras|      Jenny Rachman|         1959|             actress|
|tt0079644|      November 1828|      November 1828|    

In [ ]:
quantidade_de_linhas = df_filtered.count()
print(f"O DataFrame tem {quantidade_de_linhas} linhas.")

O DataFrame tem 46195 linhas.


In [ ]:
# Tabela temporária
df_filtered.createOrReplaceTempView("pessoas")

# Consulta SQL para contar o número de pessoas em cada geração
consulta_sql = """
SELECT *,
       2024 - anoNascimento AS idade,
       CASE
           WHEN 2024 - anoNascimento BETWEEN 1 AND 20 THEN 'juvenil'  -- Use the calculation directly
           WHEN 2024 - anoNascimento BETWEEN 21 AND 40 THEN 'adultos'
           WHEN 2024 - anoNascimento BETWEEN 41 AND 60 THEN 'maduros'
           ELSE 'idosos'
       END AS categoriaIdade
FROM pessoas;
"""

# Executa a consulta
df_filtered = spark.sql(consulta_sql)
df_filtered.show()

+---------+-------------------+-------------------+-------------+--------------------+-----------+-------------+--------------------+-------------------+-------------+--------------------+-----+--------------+
|       id|     tituloPincipal|     tituloOriginal|anoLancamento|              genero|numeroVotos|generoArtista|          personagem|        nomeArtista|anoNascimento|           profissao|idade|categoriaIdade|
+---------+-------------------+-------------------+-------------+--------------------+-----------+-------------+--------------------+-------------------+-------------+--------------------+-----+--------------+
|tt0079644|      November 1828|      November 1828|         2001|           Drama,War|        214|        actor|     Captain DeBorst|    Slamet Rahardjo|         1949|actor,director,wr...| 75.0|        idosos|
|tt0079644|      November 1828|      November 1828|         2001|           Drama,War|        214|      actress|               Laras|      Jenny Rachman|       

In [ ]:
# # Tabela temporária
# df_filtered.createOrReplaceTempView("casting")

# # Consulta SQL para contar o número de pessoas em cada geração
# consulta_sql = """
# SELECT *,
#       CAST(id AS STRING) AS id,
#       CAST(tituloPincipal AS STRING) AS tituloPrincipal,
#       CAST(tituloOriginal AS STRING) AS tituloOriginal,
#       CAST(anoLancamento AS INT) AS anoLancamento,
#       CAST(genero AS STRING) AS genero,
#       CAST(numeroVotos AS INT) AS numeroVotos,
#       CAST(generoArtista AS STRING) AS generoArtista,
#       CAST(personagem AS STRING) AS personagem,
#       CAST(nomeArtista AS STRING) AS nomeArtista,
#       CAST(anoNascimento AS INT) AS anoNascimento,
#       CAST(profissao AS STRING) AS profissao
# FROM casting;
# """

# # Executa a consulta
# df_filtered= spark.sql(consulta_sql)
# df_filtered.show()

In [ ]:
# df_filtered.printSchema()

In [ ]:
def normalize_profession_column(df_filtered):
    # Divide a coluna "profissao" em uma lista
    df_filtered = df_filtered.withColumn("profissao_lista", split(df_filtered["profissao"], ","))

    # Encontra o número máximo de valores separados na coluna "profissao"
    max_num_valores = df_filtered.select(size("profissao_lista")).agg({"size(profissao_lista)": "max"}).collect()[0][0]

    # Cria novas colunas para cada valor separado
    for i in range(1, max_num_valores + 1):
        df_filtered = df_filtered.withColumn(f"profissao{i}", col("profissao_lista").getItem(i - 1))

    # Preenche com 'não possui' quando o valor for nulo
    for i in range(1, max_num_valores + 1):
        df_filtered = df_filtered.withColumn(f"profissao{i}", when(col(f"profissao{i}").isNull(), "não possui").otherwise(col(f"profissao{i}")))

    # Elimina as colunas "profissao" e "profissao_lista"
    df_filtered = df_filtered.drop("profissao", "profissao_lista")

    return df_filtered

# Carregue seu DataFrame df_artistas aqui
df_filtered = normalize_profession_column(df_filtered)
df_filtered.show(5)


+---------+--------------+--------------+-------------+------------------+-----------+-------------+-------------------+-------------------+-------------+-----+--------------+----------+----------------+----------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|            genero|numeroVotos|generoArtista|         personagem|        nomeArtista|anoNascimento|idade|categoriaIdade|profissao1|      profissao2|profissao3|
+---------+--------------+--------------+-------------+------------------+-----------+-------------+-------------------+-------------------+-------------+-----+--------------+----------+----------------+----------+
|tt0079644| November 1828| November 1828|         2001|         Drama,War|        214|        actor|    Captain DeBorst|    Slamet Rahardjo|         1949| 75.0|        idosos|     actor|        director|    writer|
|tt0079644| November 1828| November 1828|         2001|         Drama,War|        214|      actress|              Laras|      Jenny Rachman|

In [ ]:
def normalize_genero_column(df_filtered):
    # Normalizando a coluna genero do DF das midias
    df_filtered = df_filtered.withColumn("genero_lista", split(df_filtered["genero"], ","))

    # Determine o número máximo de valores separados na coluna "profissao"
    max_num_valores = df_filtered.select(size("genero_lista")).agg({"size(genero_lista)": "max"}).collect()[0][0]

    # Crie novas colunas para cada valor separado
    for i in range(1, max_num_valores + 1):
        df_filtered = df_filtered.withColumn(f"genero{i}", col("genero_lista").getItem(i - 1))

    # Preencher com 'não possui' quando o valor for nulo
    for i in range(1, max_num_valores + 1):
        df_filtered = df_filtered.withColumn(f"genero{i}", when(col(f"genero{i}").isNull(), "não possui").otherwise(col(f"genero{i}")))

    # Eliminando "genero" e "genero_lista"
    df_filtered = df_filtered.drop("genero", "genero_lista")

    return df_filtered


# Carregue seu DataFrame df_artistas aqui
df_filtered = normalize_genero_column(df_filtered)
df_filtered.show(5)

+---------+--------------+--------------+-------------+-----------+-------------+-------------------+-------------------+-------------+-----+--------------+----------+----------------+----------+-------+--------+----------+
|       id|tituloPincipal|tituloOriginal|anoLancamento|numeroVotos|generoArtista|         personagem|        nomeArtista|anoNascimento|idade|categoriaIdade|profissao1|      profissao2|profissao3|genero1| genero2|   genero3|
+---------+--------------+--------------+-------------+-----------+-------------+-------------------+-------------------+-------------+-----+--------------+----------+----------------+----------+-------+--------+----------+
|tt0079644| November 1828| November 1828|         2001|        214|        actor|    Captain DeBorst|    Slamet Rahardjo|         1949| 75.0|        idosos|     actor|        director|    writer|  Drama|     War|não possui|
|tt0079644| November 1828| November 1828|         2001|        214|      actress|              Laras|   

In [ ]:
quantidade_de_linhas = df_filtered.count()
print(f"O DataFrame tem {quantidade_de_linhas} linhas.")

O DataFrame tem 46195 linhas.


In [ ]:
# Salvando o DataFrame particionado por anoNascimento
df_filtered.write.parquet("df_filtrado.parquet", mode="overwrite")

In [ ]:
# Abrindo o DF pessoas coletado pela API
df_artistas_json = spark.read.json("/content/pessoas.json")
df_artistas_json.show(5)

+-----+------+-------+--------------------+--------------------+---------------+---------------+----------+--------------------+
|adult|gender|     id|           known_for|known_for_department|           name|  original_name|popularity|        profile_path|
+-----+------+-------+--------------------+--------------------+---------------+---------------+----------+--------------------+
|false|     2|  73968|[{false, /4oYBCR2...|              Acting|   Henry Cavill|   Henry Cavill|    178.94|/iWdKjMry5Pt7vmxU...|
|false|     1| 544994|[{false, /gCqS4nN...|              Acting|        Vedhika|        Vedhika|   173.902|/aVCfGUDKsycQErKw...|
|false|     2|2786960|[{false, /lntyt4O...|              Acting|Gabriel Guevara|Gabriel Guevara|   149.742|/pviRYKEEmoPUfLYw...|
|false|     1|1955840|[{false, /vh9Rk0B...|              Acting| Afra Saraçoğlu| Afra Saraçoğlu|   144.346|/qxYuc8I8q9PyxU1L...|
|false|     2|    976|[{false, /ibKeXah...|              Acting|  Jason Statham|  Jason Statham| 

In [ ]:
# Tabela temporária
df_artistas_json.createOrReplaceTempView("casting")

# Consulta SQL para contar o número de pessoas em cada geração
consulta_sql = """
SELECT *,
       CAST(id AS STRING) AS id_avaliacao
FROM casting;
"""

# Executa a consulta
df_filtered_artistas = spark.sql(consulta_sql)
df_filtered_artistas.show()

+-----+------+-------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+------------+
|adult|gender|     id|           known_for|known_for_department|                name|       original_name|popularity|        profile_path|id_avaliacao|
+-----+------+-------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+------------+
|false|     2|  73968|[{false, /4oYBCR2...|              Acting|        Henry Cavill|        Henry Cavill|    178.94|/iWdKjMry5Pt7vmxU...|       73968|
|false|     1| 544994|[{false, /gCqS4nN...|              Acting|             Vedhika|             Vedhika|   173.902|/aVCfGUDKsycQErKw...|      544994|
|false|     2|2786960|[{false, /lntyt4O...|              Acting|     Gabriel Guevara|     Gabriel Guevara|   149.742|/pviRYKEEmoPUfLYw...|     2786960|
|false|     1|1955840|[{false, /vh9Rk0B...|              Acting|      Afra Saraçoğlu|   

In [ ]:
# Cria a visualização temporária
df_filtered_artistas.createOrReplaceTempView("remover_coluna")

# Consulta SQL
consulta_sql = '''
SELECT name, original_name, popularity, id_avaliacao
FROM remover_coluna;
'''

# Executa a consulta
df_filtered_artistas = spark.sql(consulta_sql)
df_filtered_artistas.show()

+--------------------+--------------------+----------+------------+
|                name|       original_name|popularity|id_avaliacao|
+--------------------+--------------------+----------+------------+
|        Henry Cavill|        Henry Cavill|    178.94|       73968|
|             Vedhika|             Vedhika|   173.902|      544994|
|     Gabriel Guevara|     Gabriel Guevara|   149.742|     2786960|
|      Afra Saraçoğlu|      Afra Saraçoğlu|   144.346|     1955840|
|       Jason Statham|       Jason Statham|   138.873|         976|
|          Ranjan Raj|          Ranjan Raj|   128.198|     2318714|
|   Supaporn Malisorn|   Supaporn Malisorn|   123.835|     3878062|
|  Scarlett Johansson|  Scarlett Johansson|   119.322|        1245|
|     Anya Taylor-Joy|     Anya Taylor-Joy|   117.192|     1397778|
|        Ana de Armas|        Ana de Armas|   109.767|      224513|
|      Sydney Sweeney|      Sydney Sweeney|    97.889|      115440|
|       Kevin Costner|       Kevin Costner|    9

In [ ]:
df_avaliacao_atores = df_filtered.join(df_filtered_artistas, df_filtered["nomeArtista"] == df_filtered_artistas["name"])
df_avaliacao_atores.show(20)

+---------+-------------------+-------------------+-------------+-----------+-------------+--------------------+--------------------+-------------+-----+--------------+----------+------------------+---------------+-------+--------+----------+--------------------+--------------------+----------+------------+
|       id|     tituloPincipal|     tituloOriginal|anoLancamento|numeroVotos|generoArtista|          personagem|         nomeArtista|anoNascimento|idade|categoriaIdade|profissao1|        profissao2|     profissao3|genero1| genero2|   genero3|                name|       original_name|popularity|id_avaliacao|
+---------+-------------------+-------------------+-------------+-----------+-------------+--------------------+--------------------+-------------+-----+--------------+----------+------------------+---------------+-------+--------+----------+--------------------+--------------------+----------+------------+
|tt0116391|               Gang|               Gang|         2000|        

In [ ]:
# Checando a quantidade de linhas
quantidade_de_linhas_join = df_avaliacao_atores.count()
print(f"O DataFrame artistas tem {quantidade_de_linhas_join} linhas.")

O DataFrame artistas tem 12497 linhas.


In [ ]:
df_avaliacao_atores.write.parquet("df_avaliacao_atores.parquet", mode="overwrite", partitionBy="categoriaIdade")